In [4]:
!pip -q install pypdf faiss-cpu langchain_community langchain_text_splitters langchain_openai langchain_huggingface tavily-python ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.2 MB/s eta 0:00:00a 0:00:01


In [5]:
import os
import re
import json
from getpass import getpass
from urllib.parse import urlparse
from typing import TypedDict, List, Dict, Any

import pandas as pd
from IPython.display import display, Markdown

from tavily import TavilyClient
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

### Set configs

In [6]:
MODEL_NAME = "google/gemini-2.5-flash"
SEARCHES_PER_TOPIC = 4
RESULTS_PER_QUERY = 5
MAX_SOURCES = 6
MAX_CHARS_PER_PAGE = 7000

### Define structured output schemas

In [8]:
class SearchPlan(BaseModel):
    refined_topic: str = Field(description="Cleaner and sharper version of the topic")
    sub_queries: List[str] = Field(description="Search-friendly sub-queries for web research")


class URLSelection(BaseModel):
    selected_ids: List[int] = Field(description="1-based row ids of the best sources to keep")
    

class PageSummary(BaseModel):
    summary: str = Field(description="Short summary of the page")
    top_points: List[str] = Field(description="Most useful facts or claims from the page")
    risks: List[str] = Field(description="Possible risks, limitations, uncertainty, or bias from the page")
    

class SourceItem(BaseModel):
    title: str
    url: str
    why_it_matters: str

class ResearchReport(BaseModel):
    topic: str
    executive_summary: str
    top_findings: List[str]
    risks: List[str]
    sources : List[SourceItem]

### Create all helper functions

In [33]:
def clean_text(text, limit=None):
    text = re.sub(r"\s+", " ", str(text)).strip()
    if limit is not None:
        return text[:limit]
    return text

def get_domain(url):
    try:
        return urlparse(url).netloc.replace("www.", "").strip()
    except:
        return ""

def guess_tavily_topic(topic):
    text = topic.lower()
    news_words = [
        "latest", "recent", "current", "today", "news", "update", "updates",
        "this week", "this month", "2025", "2026", "trend", "trends"
    ]
    if any(word in text for word in news_words):
        return "news"
    return "general"

def dedupe_rows(rows):
    best = {}
    for row in rows:
        url = row["url"]
        if url not in best or row["score"] > best[url]["score"]:
            best[url] = row
        
        return sorted(best.values(), key=lambda x: x["score"], reverse=True)

In [ ]:
def make_search_df(rows):
    if not rows:
        return pd.DataFrame(columns=["query", "title", "domain", "score", "url", "snippet"])
    df = pd.DataFrame(rows)
    return df[["query", "title", "domain", "score", "url", "snippet"]]

def make_filtered_df(rows):
    if not rows:
        return pd.DataFrame(columns=["title", "domain", "score", "url", "snippet"])
    df = pd.DataFrame(rows)
    return df[["title", "domain", "score", "url", "snippet"]]

def make_summary_df(rows):
    if not rows:
        return pd.DataFrame(columns=["title", "url", "summary", "top_points", "risks"])
    data = []
    for row in rows:
        data.append(
            {
                "title": row["title"],
                "url": row["url"],
                "summary": row["summary"],
                "top_points": " | ".join(row["top_points"]),
                "risks": " | ".join(row["risks"])
            }
        )
    return pd.DataFrame(data)

In [36]:
def build_markdown_report(report):
    lines = []
    lines.append(f"# Research Report: {report.topic}")
    lines.append("")
    lines.append("## Executive Summary")
    lines.append(report.executive_summary)
    lines.append("")
    lines.append("## Top Findings")
    
    for item in report.top_findings:
        lines.append(f"- {item}")
    lines.append("")
    lines.append("## Risks")
    if report.risks:
        for item in report.risks:
            lines.append(f"- {item}")
    else:
        lines.append("- No major risks were identified from the selected sources.")
    lines.append("")
    lines.append("## Sources")
    
    for i, source in enumerate(report.sources, start=1):
        lines.append(f"{i}. [{source.title}]({source.url}) — {source.why_it_matters}")
    return "\n".join(lines)

### Define workflow state

In [37]:
class ResearchState(TypedDict):
    topic: str
    tavily_topic: str
    sub_queries: List[str]
    search_rows = List[Dict[str, Any]]
    filtered_rows: List[Dict[str, Any]]
    extracted_rows: List[Dict[str, Any]]
    page_summaries: List[Dict[str, Any]]
    final_report : Dict[str, Any]
    report_markdown = str

### Create graph nodes